In [1]:
import os
import json
import random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

# Configuration
DATA_DIR = "florence_dataset"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
ANNOTATION_FILE = os.path.join(DATA_DIR, "annotations.jsonl")
OUTPUT_DIR = "./florence2-grid-finetuned"
MODEL_NAME = "microsoft/Florence-2-base"
GRID_SIZE = 10
NUM_SAMPLES = 100  # consider reducing for faster generation/training

# Detect device (MPS for Apple Silicon, CUDA, or CPU)
device = torch.device("mps") if torch.backends.mps.is_available() else (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {device}")

# Global processor (initialized in main)
processor = None

def generate_dataset():
    """
    Generate synthetic grid images and QA annotations.
    """
    import matplotlib.pyplot as plt
    os.makedirs(IMAGE_DIR, exist_ok=True)
    with open(ANNOTATION_FILE, 'w') as f:
        for idx in range(NUM_SAMPLES):
            # Create grid with obstacles and target
            grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
            obstacle_positions = random.sample(
                [(i, j) for i in range(GRID_SIZE) for j in range(GRID_SIZE)],
                random.randint(5, 15)
            )
            for pos in obstacle_positions:
                grid[pos] = 1
            empty = [(i, j) for i in range(GRID_SIZE) for j in range(GRID_SIZE) if grid[i, j] == 0]
            target_position = random.choice(empty)
            grid[target_position] = 2

            # Render and save image
            fig, ax = plt.subplots(figsize=(4, 4))
            for i in range(GRID_SIZE):
                for j in range(GRID_SIZE):
                    color = "white" if grid[i, j] == 0 else ("black" if grid[i, j] == 1 else "red")
                    ax.add_patch(plt.Rectangle([j, GRID_SIZE - i - 1], 1, 1, facecolor=color, edgecolor='gray'))
            ax.set_xlim(0, GRID_SIZE)
            ax.set_ylim(0, GRID_SIZE)
            ax.axis('off')
            image_path = os.path.join(IMAGE_DIR, f"grid_{idx}.png")
            fig.savefig(image_path, bbox_inches='tight')
            plt.close(fig)

            # Write annotations
            anns = [
                {"image": image_path, "question": "Where is the red target?", "answer": f"The target is at position {target_position}."},
                {"image": image_path, "question": "Which cells are obstacles?", "answer": f"Obstacles are at positions {obstacle_positions}."}
            ]
            for ann in anns:
                f.write(json.dumps(ann) + '\n')

class GridDataset(Dataset):
    """
    Dataset for 2D grid images and QA pairs.
    Expects JSONL with image, question, answer.
    """
    def __init__(self, annotation_file, proc):
        self.samples = [json.loads(l) for l in open(annotation_file)]
        self.processor = proc

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['image']).convert("RGB")
        inputs = self.processor(text=sample['question'], images=image, return_tensors="pt", padding=True)
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs['answer_text'] = sample['answer']
        return inputs

def data_collator(features):
    """Collate and pad inputs and labels."""
    global processor
    answers = [f.pop('answer_text') for f in features]
    tokenized = processor.tokenizer(answers, padding=True, return_tensors='pt')
    labels = tokenized.input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # Pad input_ids and attention_mask
    ids = [f['input_ids'] for f in features]
    masks = [f['attention_mask'] for f in features]
    pad_id = processor.tokenizer.pad_token_id
    max_len = max([x.shape[0] for x in ids])
    padded_ids = torch.stack([torch.cat([x, torch.full((max_len-x.shape[0],), pad_id, dtype=torch.long)]) for x in ids])
    padded_masks = torch.stack([torch.cat([m, torch.zeros((max_len-m.shape[0],), dtype=torch.long)]) for m in masks])

    pixel_values = torch.stack([f['pixel_values'] for f in features])
    return {
        'pixel_values': pixel_values.to(device),
        'input_ids': padded_ids.to(device),
        'attention_mask': padded_masks.to(device),
        'labels': labels.to(device)
    }

def main():
    global processor
    # Generate dataset if needed
    if not os.path.exists(ANNOTATION_FILE):
        print('Generating dataset...')
        generate_dataset()

    # Load model and processor
    print('Loading model and processor...')
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model.to(device)

    # Enable efficient training
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    if hasattr(torch, 'compile'):
        model = torch.compile(model)

    # PEFT LoRA for parameter-efficient fine-tuning
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj']
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # Prepare dataset
    train_dataset = GridDataset(ANNOTATION_FILE, processor)

    # Training args: fewer epochs, bigger batch if possible
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        dataloader_num_workers=os.cpu_count()//2 or 2,
        num_train_epochs=1,
        save_steps=1000,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        bf16=torch.backends.mps.is_available(),
        remove_unused_columns=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator
    )

    # Train
    print('Starting training...')
    trainer.train()

    # Save
    print('Saving model...')
    model.save_pretrained(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print('Done.')

if __name__ == '__main__':
    main()

/Users/shahmirkhan/miniconda3/envs/torch-gpu/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/shahmirkhan/miniconda3/envs/torch-gpu/lib/python3.8/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <AA076682-B999-31A8-AEA6-2BF697B54762> /Users/shahmirkhan/miniconda3/envs/torch-gpu/lib/python3.8/site-packages/torchvision/image.so
  Expected in:     <AEDB2D9B-AE02-3964-90EC-49E2AD5A10A1> /Users/shahmirkhan/miniconda3/envs/torch-gpu/lib/python3.8/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/Users/shahmirkhan/miniconda3/envs/torch-gpu/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: U

Using device: mps
Generating dataset...
Loading model and processor...


/Users/shahmirkhan/miniconda3/envs/torch-gpu/lib/python3.8/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


trainable params: 442,368 || all params: 271,246,336 || trainable%: 0.1631
Starting training...


  0%|          | 0/25 [00:00<?, ?it/s]

KeyboardInterrupt: 